# Multi-Agent Systems with CrewAI

**CrewAI** is an open-source framework for orchestrating **role-playing, autonomous AI agents**. Instead of defining graphs and state machines (like LangGraph), you define agents with **roles, goals, and backstories** -- and let them collaborate as a crew.

## What This Demo Covers

1. **CrewAI vs LangChain/LangGraph** -- Philosophy, pros, cons, and side-by-side comparison
2. **Core Concepts** -- Agents, Tasks, Tools, Crews, and Processes
3. **Example 1: Sequential Process** -- Research & Writing pipeline
4. **Example 2: Hierarchical Process** -- Manager delegates to specialists
5. **Example 3: Agent Delegation** -- Agents hand off work to each other

---

## How CrewAI Works (Mental Model)

Think of it like assembling a team at a company:

| Concept | Real-World Analogy |
|---------|-------------------|
| **Agent** | A team member with a job title, expertise, and personality |
| **Task** | A work item assigned to a team member |
| **Tool** | Software or resources the team member can use |
| **Crew** | The team itself |
| **Process** | How the team organizes work (waterfall vs. managed) |

## CrewAI vs LangChain/LangGraph -- Detailed Comparison

### Philosophy

| | CrewAI | LangChain / LangGraph |
|---|---|---|
| **Core idea** | Role-playing agents with goals & backstories | Composable chains (LangChain) and stateful graphs (LangGraph) |
| **Multi-agent** | First-class -- agents, tasks, crews are the primitives | Built manually via graph nodes and conditional edges |
| **Control flow** | Implicit (process type + delegation) | Explicit (you wire every edge and condition) |
| **Abstraction level** | High -- describe *what* each agent does | Low-to-medium -- define *how* data flows |

### Pros & Cons

#### CrewAI Pros
- **Simpler multi-agent setup** -- a Crew with 3 agents and 3 tasks is ~30 lines of code
- **Role-based design** -- `role`, `goal`, `backstory` make agent behavior intuitive
- **Built-in delegation** -- agents can hand off work to each other without extra wiring
- **Sequential & hierarchical processes** out of the box
- **Less boilerplate** -- no need to define `TypedDict` state schemas or graph edges
- **Standalone framework** -- built from scratch, lightweight, no heavy dependencies

#### CrewAI Cons
- **Less fine-grained control** -- you can't define arbitrary conditional edges or loops
- **Smaller ecosystem** -- fewer integrations than LangChain's 100+
- **Less mature** -- newer project, still evolving
- **Limited observability** -- no equivalent to LangSmith's deep tracing (though improving)
- **Opinionated** -- harder to break out of the agent/task/crew paradigm

#### LangChain / LangGraph Pros
- **Massive ecosystem** -- 100+ model, vector store, and tool integrations
- **Deep observability** -- LangSmith provides production-grade tracing and evaluation
- **Full graph control** -- define any workflow topology (loops, branches, fan-out)
- **Production-proven** -- used by thousands of companies
- **State management** -- built-in checkpointing and persistence in LangGraph
- **Human-in-the-loop** -- first-class interrupt and approval support

#### LangChain / LangGraph Cons
- **More verbose for multi-agent** -- supervisor pattern requires manual routing logic
- **Steeper learning curve** -- StateGraph, TypedDict, conditional edges, etc.
- **Heavier abstractions** -- more layers between you and the LLM
- **Breaking changes** -- fast-moving project, APIs evolve frequently

### Side-by-Side Code Comparison

**Creating a 2-agent research pipeline:**

| | CrewAI | LangGraph |
|---|---|---|
| Define agents | `Agent(role=..., goal=..., backstory=...)` | Define node functions + prompts |
| Define tasks | `Task(description=..., agent=...)` | Encode in state + node logic |
| Wire flow | `Crew(process=Process.sequential)` | `add_edge(START, "node1")`, `add_edge("node1", "node2")`, ... |
| Run | `crew.kickoff()` | `app.invoke({...})` |
| **Lines of code** | ~30 | ~60-80 |

### When to Use Which

| Scenario | Recommendation |
|----------|---------------|
| Quick multi-agent prototype | **CrewAI** -- faster to set up |
| Complex branching/looping workflows | **LangGraph** -- explicit control flow |
| Role-based teams (researcher, writer, editor) | **CrewAI** -- natural fit |
| Production system with deep observability | **LangGraph + LangSmith** |
| Need 100+ integrations | **LangChain** ecosystem |
| Human-in-the-loop approvals | **LangGraph** -- built-in interrupt support |
| Simple delegation between agents | **CrewAI** -- `allow_delegation=True` |

## Setup & Installation

CrewAI requires **Python >= 3.10**.

```bash
pip install crewai crewai-tools python-dotenv
```

The cell below installs dependencies (if needed) and loads your `OPENAI_API_KEY` from the `.env` file. **Run it first before any other code cell.**

In [ ]:
# Uncomment to install if needed
# !pip install crewai crewai-tools python-dotenv

import os
from dotenv import load_dotenv

# Try multiple .env locations (langgraph demo has the shared keys)
for env_path in [
    os.path.join("..", "langchain-langgraph-demo", ".env"),
    os.path.join("..", ".env"),
    ".env",
]:
    if os.path.exists(env_path):
        load_dotenv(env_path, override=True)
        if "OPENAI_API_KEY" in os.environ:
            print(f"OPENAI_API_KEY loaded from {env_path} (ends with ...{os.environ['OPENAI_API_KEY'][-4:]})")
            break

assert "OPENAI_API_KEY" in os.environ, (
    "OPENAI_API_KEY not found! Add it to one of:\n"
    "  - ../langchain-langgraph-demo/.env\n"
    "  - ../.env\n"
    "  - .env"
)

---

## Core Concept 1: Agents

An **Agent** is defined by three key attributes:

| Attribute | Purpose |
|-----------|--------|
| `role` | Job title -- defines what the agent specializes in |
| `goal` | What the agent is trying to achieve |
| `backstory` | Context and personality -- guides the agent's behavior |

Additional options:
- `llm` -- which model to use (default: `gpt-4`)
- `tools` -- list of tools the agent can call
- `verbose` -- print detailed execution logs
- `allow_delegation` -- let the agent hand off work to other agents in the crew
- `memory` -- retain context across tasks
- `max_iter` -- max reasoning iterations (default: 20)

In [ ]:
from crewai import Agent

# Agent 1: Researcher
researcher = Agent(
    role="Senior Research Analyst",
    goal="Find comprehensive and accurate information on the given topic",
    backstory=(
        "You are a seasoned research analyst with 15 years of experience. "
        "You excel at finding key facts, identifying trends, and organizing "
        "information in a clear, structured way. You always cite your reasoning."
    ),
    verbose=True,
    allow_delegation=False,
    llm="gpt-4o"
)

# Agent 2: Writer
writer = Agent(
    role="Content Writer",
    goal="Create engaging, well-structured content from research findings",
    backstory=(
        "You are a professional content writer who transforms raw research "
        "into polished, reader-friendly articles. You focus on clarity, "
        "structure, and making complex topics accessible."
    ),
    verbose=True,
    allow_delegation=False,
    llm="gpt-4o"
)

# Agent 3: Editor
editor = Agent(
    role="Senior Editor",
    goal="Review and improve content for quality, accuracy, and readability",
    backstory=(
        "You are a meticulous editor with a sharp eye for detail. "
        "You check for factual accuracy, logical flow, grammar, and "
        "ensure the final output meets professional standards."
    ),
    verbose=True,
    allow_delegation=False,
    llm="gpt-4o"
)

print("Created 3 agents:")
print(f"  1. {researcher.role}")
print(f"  2. {writer.role}")
print(f"  3. {editor.role}")

---

## Core Concept 2: Tasks

A **Task** defines a specific piece of work for an agent. Key parameters:

| Parameter | Required | Purpose |
|-----------|----------|--------|
| `description` | Yes | What needs to be done |
| `expected_output` | Yes | What the result should look like |
| `agent` | No | Which agent handles this task |
| `context` | No | List of other tasks whose outputs feed into this one |
| `output_file` | No | Save the result to a file |
| `output_pydantic` | No | Enforce structured output via a Pydantic model |
| `human_input` | No | Ask for human review before finalizing |

In [ ]:
from crewai import Task

# Task 1: Research
research_task = Task(
    description=(
        "Research the topic: 'The current state of AI agents in 2025'. "
        "Find key developments, major frameworks, industry adoption trends, "
        "and challenges. Focus on practical, factual information."
    ),
    expected_output=(
        "A structured research summary with sections: "
        "Key Developments, Major Frameworks, Industry Adoption, "
        "and Challenges. Each section should have 3-5 bullet points."
    ),
    agent=researcher
)

# Task 2: Write -- uses research_task output as context
writing_task = Task(
    description=(
        "Using the research findings provided, write a concise article "
        "about AI agents in 2025. The article should be informative, "
        "well-organized, and accessible to a technical audience."
    ),
    expected_output=(
        "A 300-500 word article with a title, introduction, "
        "3-4 body sections, and a conclusion."
    ),
    agent=writer,
    context=[research_task]  # Writer receives the researcher's output
)

# Task 3: Edit -- uses writing_task output as context
editing_task = Task(
    description=(
        "Review the article for clarity, accuracy, grammar, and structure. "
        "Provide the final polished version of the article. "
        "Fix any issues and improve readability."
    ),
    expected_output=(
        "The final, polished article ready for publication. "
        "Include a brief editor's note listing any changes made."
    ),
    agent=editor,
    context=[writing_task]  # Editor receives the writer's output
)

print("Created 3 tasks with context chain:")
print("  research_task --> writing_task --> editing_task")
print("  (each task feeds its output to the next)")

---

## Core Concept 3: Tools

Tools give agents the ability to interact with the outside world. CrewAI supports:

1. **`crewai_tools`** -- built-in tools (web search, file read, scraping, etc.)
2. **Custom tools** -- define your own using the `@tool` decorator
3. **LangChain tools** -- any LangChain-compatible tool works too

### Custom Tool Example

In [ ]:
from crewai.tools import tool

@tool("Word Counter")
def word_counter(text: str) -> str:
    """Count the number of words in the given text."""
    count = len(text.split())
    return f"The text contains {count} words."

@tool("Summarizer")
def summarizer(text: str) -> str:
    """Create a one-line summary of the given text."""
    # In production, this could call an LLM or use extractive summarization
    words = text.split()
    if len(words) > 20:
        return "Summary: " + " ".join(words[:20]) + "..."
    return "Summary: " + text

print("Created custom tools:")
print(f"  - word_counter: Count the number of words in the given text.")
print(f"  - summarizer: Create a one-line summary of the given text.")

---

## Example 1: Sequential Process (Research & Writing Crew)

**Sequential process** runs tasks one after another in order. Each task can use the output of previous tasks via the `context` parameter.

```
Researcher --> Writer --> Editor --> Final Article
```

This is equivalent to the **sequential workflow** in LangGraph (Demo 5, Section 2), but requires significantly less code.

### LangGraph equivalent
In LangGraph you would:
1. Define a `TypedDict` state schema
2. Write 3 node functions
3. Build a `StateGraph`, add nodes, add edges
4. Compile and invoke

### CrewAI version
Just create agents, tasks, and a crew -- done.

In [ ]:
from crewai import Crew, Process

# Assemble the crew with sequential process
sequential_crew = Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, writing_task, editing_task],
    process=Process.sequential,  # Tasks run in order
    verbose=True
)

print("Crew assembled!")
print(f"  Agents: {[a.role for a in sequential_crew.agents]}")
print(f"  Tasks:  {len(sequential_crew.tasks)}")
print(f"  Process: {sequential_crew.process}")

In [ ]:
# Run the crew
result = sequential_crew.kickoff()

print("=" * 80)
print("FINAL OUTPUT")
print("=" * 80)
print(result.raw)

In [ ]:
# Inspect individual task outputs
print("=" * 80)
print("TASK-BY-TASK RESULTS")
print("=" * 80)

task_labels = ["Research", "Writing", "Editing"]
for i, task_output in enumerate(result.tasks_output):
    label = task_labels[i] if i < len(task_labels) else f"Task {i+1}"
    print(f"\n--- {label} ({len(task_output.raw)} chars) ---")
    print(task_output.raw[:500])  # First 500 chars
    if len(task_output.raw) > 500:
        print("...")

# Token usage
print("\n--- Token Usage ---")
print(result.token_usage)

---

## Example 2: Hierarchical Process (Manager Pattern)

**Hierarchical process** introduces a **manager agent** that automatically:
1. Reads the tasks
2. Decides which agent should handle each task
3. Delegates work and validates results

```
                        +-> Data Analyst
Manager (auto-created) -+-> Strategist
                        +-> Report Writer
```

### Comparison with LangGraph Supervisor (Demo 5, Section 3-4)

| | CrewAI Hierarchical | LangGraph Supervisor |
|---|---|---|
| Manager agent | Auto-created OR custom | You build it from scratch |
| Routing logic | Implicit (manager decides) | Explicit (structured output + conditional edges) |
| Setup effort | Set `process=Process.hierarchical` | Define supervisor prompt, Pydantic model, router function |
| Control | Less (trust the manager) | Full (you define every route) |

In [ ]:
# Create specialist agents for a business analysis crew
data_analyst = Agent(
    role="Data Analyst",
    goal="Analyze data and identify key metrics and trends",
    backstory=(
        "You are a data analyst who excels at extracting insights from data. "
        "You focus on numbers, trends, and quantitative analysis."
    ),
    verbose=True,
    llm="gpt-4o"
)

strategist = Agent(
    role="Business Strategist",
    goal="Develop actionable business strategies based on analysis",
    backstory=(
        "You are a seasoned business strategist who turns data insights "
        "into actionable recommendations. You think about market positioning, "
        "competitive advantages, and growth opportunities."
    ),
    verbose=True,
    llm="gpt-4o"
)

report_writer = Agent(
    role="Report Writer",
    goal="Create clear, professional business reports",
    backstory=(
        "You are an expert at creating executive-level reports that "
        "combine data analysis with strategic recommendations into "
        "a coherent, actionable document."
    ),
    verbose=True,
    llm="gpt-4o"
)

print("Created 3 specialist agents for hierarchical crew")

In [ ]:
# Define tasks -- note: in hierarchical mode, the manager decides
# which agent handles which task, but we can still suggest assignments

analysis_task = Task(
    description=(
        "Analyze the current state of the AI SaaS market. "
        "Identify the top 3 trends, market size estimates, "
        "and key players. Use available data and reasoning."
    ),
    expected_output=(
        "A data-driven analysis with: top 3 trends (with evidence), "
        "market size estimates, and a list of 5 key players with "
        "their differentiators."
    ),
    agent=data_analyst
)

strategy_task = Task(
    description=(
        "Based on the market analysis, develop 3 strategic "
        "recommendations for a startup entering the AI SaaS space. "
        "Consider competitive positioning, target segments, "
        "and go-to-market approach."
    ),
    expected_output=(
        "3 strategic recommendations, each with: description, "
        "rationale, target segment, and estimated impact."
    ),
    agent=strategist,
    context=[analysis_task]
)

report_task = Task(
    description=(
        "Compile the analysis and strategy into a professional "
        "executive report. Include an executive summary, "
        "market analysis section, strategic recommendations, "
        "and next steps."
    ),
    expected_output=(
        "A professional executive report with sections: "
        "Executive Summary, Market Analysis, Strategic Recommendations, "
        "and Next Steps. Should be 400-600 words."
    ),
    agent=report_writer,
    context=[analysis_task, strategy_task]
)

print("Created 3 tasks for hierarchical crew")

In [ ]:
# Assemble a hierarchical crew
# The manager_llm powers the auto-created manager agent that coordinates work

hierarchical_crew = Crew(
    agents=[data_analyst, strategist, report_writer],
    tasks=[analysis_task, strategy_task, report_task],
    process=Process.hierarchical,  # Manager auto-delegates
    manager_llm="gpt-4o",
    verbose=True
)

print(f"Hierarchical crew assembled!")
print(f"  Process: {hierarchical_crew.process}")
print(f"  Manager LLM will coordinate {len(hierarchical_crew.agents)} agents")

In [ ]:
# Run the hierarchical crew
hierarchical_result = hierarchical_crew.kickoff()

print("=" * 80)
print("HIERARCHICAL CREW -- FINAL REPORT")
print("=" * 80)
print(hierarchical_result.raw)

---

## Example 3: Agent Delegation

When `allow_delegation=True`, an agent can decide to **hand off part of its work** to another agent in the crew. This is a powerful pattern for complex tasks where one agent realizes it needs help.

```
Lead Developer (allow_delegation=True)
    |
    +-- "I need code review" --> Code Reviewer
    +-- "I need tests" -------> QA Engineer
```

### How it differs from LangGraph

In LangGraph, delegation requires explicit conditional edges and router functions. In CrewAI, the agent autonomously decides when and what to delegate -- no extra wiring needed.

In [ ]:
# Create agents with delegation enabled

lead_developer = Agent(
    role="Lead Developer",
    goal="Design and implement a solution, delegating specialized tasks as needed",
    backstory=(
        "You are a lead developer who oversees the full development process. "
        "You write the initial solution but know when to ask specialists "
        "for help with code review or testing."
    ),
    verbose=True,
    allow_delegation=True,  # Can delegate to other agents
    llm="gpt-4o"
)

code_reviewer = Agent(
    role="Code Reviewer",
    goal="Review code for bugs, security issues, and best practices",
    backstory=(
        "You are a meticulous code reviewer who catches bugs and security "
        "vulnerabilities. You suggest improvements for code quality, "
        "performance, and maintainability."
    ),
    verbose=True,
    allow_delegation=False,
    llm="gpt-4o"
)

qa_engineer = Agent(
    role="QA Engineer",
    goal="Write test cases and verify code correctness",
    backstory=(
        "You are a QA engineer focused on test coverage and edge cases. "
        "You write unit tests, integration tests, and verify that "
        "code handles edge cases properly."
    ),
    verbose=True,
    allow_delegation=False,
    llm="gpt-4o"
)

print("Created delegation-enabled team:")
print(f"  {lead_developer.role} (can delegate)")
print(f"  {code_reviewer.role}")
print(f"  {qa_engineer.role}")

In [ ]:
# Single task that the lead developer may delegate parts of
development_task = Task(
    description=(
        "Develop a Python function that validates email addresses. "
        "The function should handle common edge cases. "
        "Make sure the code is reviewed and tested."
    ),
    expected_output=(
        "A complete Python solution including: "
        "1) The email validation function, "
        "2) Code review feedback, "
        "3) Test cases covering normal and edge cases."
    ),
    agent=lead_developer
)

# The crew includes all agents so the lead can delegate to them
dev_crew = Crew(
    agents=[lead_developer, code_reviewer, qa_engineer],
    tasks=[development_task],
    process=Process.sequential,
    verbose=True
)

dev_result = dev_crew.kickoff()

print("=" * 80)
print("DELEGATION RESULT")
print("=" * 80)
print(dev_result.raw)

---

## Summary & Key Takeaways

### What We Covered

| Section | What You Learned |
|---------|------------------|
| Comparison | CrewAI = role-based simplicity, LangGraph = graph-level control |
| Agents | Define specialists with `role`, `goal`, `backstory` |
| Tasks | Work items with `description`, `expected_output`, and `context` chaining |
| Tools | Custom functions agents can call via `@tool` decorator |
| Sequential | Tasks run in order, each building on previous output |
| Hierarchical | Manager agent auto-delegates to specialists |
| Delegation | Agents autonomously hand off work via `allow_delegation=True` |

### Decision Guide

```
Do you need multi-agent collaboration?
  |
  +-- No --> Use LangChain (single agent is fine)
  |
  +-- Yes --> Do you need fine-grained control over the workflow?
                |
                +-- Yes --> Use LangGraph
                |           (explicit edges, state, loops, human-in-the-loop)
                |
                +-- No --> Are your agents role-based specialists?
                            |
                            +-- Yes --> Use CrewAI
                            |           (agents, tasks, crews -- fast to build)
                            |
                            +-- No --> Use LangGraph
                                       (more flexible for custom patterns)
```

### Best Practices

1. **Start with sequential** -- add hierarchy only when you need dynamic delegation
2. **Keep agents focused** -- one role per agent, clear goals
3. **Use context chaining** -- `context=[previous_task]` to pass outputs forward
4. **Monitor costs** -- each agent makes LLM calls; more agents = more tokens
5. **Be specific in descriptions** -- vague tasks produce vague results
6. **Set `verbose=True` during development** -- turn it off in production

### Resources

- [CrewAI Documentation](https://docs.crewai.com)
- [CrewAI GitHub](https://github.com/crewAIInc/crewAI)
- [LangGraph Multi-Agent Tutorial](https://langchain-ai.github.io/langgraph/tutorials/multi_agent/) (for comparison)
- [Demo 5: Multi-Agent Systems with LangGraph](../langchain-langgraph-demo/Demo_5_Multi_Agent_Systems.ipynb) (our LangGraph version)